# 3D coasts: useful depth, dangerous certainty

**Time:** 40 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Build a transparent terrain-and-water scene; distinguish elevation from geographic stems; explore sensitivity without implying hydrodynamic realism.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/07_three_dimensions.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [1]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

Ready: public poster aggregates + clearly labeled fictional facility fixtures.


## What does height represent?

A three-dimensional display can encode physical elevation, a statistical value or time. Those meanings are not interchangeable. A map with stems whose height is the percentage of exposed facilities is a geographic chart, not a digital elevation model. A space–time cube uses its vertical axis for time.

The fictional terrain below is an analytic surface over a 4 × 4 km square. It is independent of the city fixtures and has no real coordinate reference system. The water plane is a threshold illustration. It does not simulate water flowing around barriers, wave overtopping, tides, drainage or river flooding.

In [2]:
x=np.linspace(0,4,51);y=np.linspace(0,4,51)
X,Y=np.meshgrid(x,y)
terrain=.15+.48*X+.16*np.sin(2*X+Y)+.08*np.cos(3*Y)
water=.7+toy_slr('RCP 8.5',2100)
fig=go.Figure([go.Surface(x=x,y=y,z=terrain,colorscale='Earth',showscale=False,name='Invented terrain'),
              go.Surface(x=x,y=y,z=np.full_like(terrain,water),opacity=.55,colorscale=[[0,'#229bd1'],[1,'#229bd1']],showscale=False)])
fig.update_layout(title='Independent fictional coast — exaggerated vertical scale',
    scene=dict(xaxis_title='x (km)',yaxis_title='y (km)',zaxis_title='Toy height (m)',aspectratio=dict(x=1,y=1,z=.6)))
show(fig)

<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="b62260fb-490a-486f-97d0-73087ba6cae7" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("b62260fb-490a-486f-97d0-73087ba6cae7")) { Plotly.newPlot( "b62260fb-490a-486f-97d0-73087ba6cae7", [{"colorscale":[[0.0,"rgb(161, 105, 40)"],[0.16666666666666666,"rgb(189, 146, 90)"],[0.3333333333333333,"rgb(214, 189, 141)"],[0.5,"rgb(237, 234, 194)"],[0.6666666666666666,"rgb(181, 200, 184)"],[0.8333333333333334,"rgb(121, 167, 172)"],[1.0,"rgb(40, 135, 161)"]],"name":"Invented terrain","showscale":false,"x":{"dtype":"f8","bdata":"AAAAAAAAAAB7FK5H4Xq0P3sUrkfhesQ\u002fuB6F61G4zj97FK5H4XrUP5qZmZmZmdk\u002fuB6F61G43j\u002fsUbgehevhP3sUrkfheuQ\u002fCtejcD0K5z+amZmZmZnpPylcj8L1KOw\u002fuB6F61G47j+kcD0K16PwP+xRuB6F6\u002fE\u002fMzMzMzMz8z97FK5H4Xr0P8P1KFyPwvU\u002fCtejcD0K9z9SuB6F61H4P5qZmZmZmfk\u002f4XoUrkfh+j8pXI\u002fC9Sj8P3E9CtejcP0\u002fuB6F61G4\u002fj8AAAAAAAAAQKRwPQrXowBASOF6FK5HAUDsUbgehesBQI\u002fC9ShcjwJAMzMzMzMzA0DXo3A9CtcDQHsUrkfhegRAH4XrUbgeBUDD9Shcj8IFQGdmZmZmZgZACtejcD0KB0CuR+F6FK4HQFK4HoXrUQhA9ihcj8L1CECamZmZmZkJQD4K16NwPQpA4XoUrkfhCkCF61G4HoULQClcj8L1KAxAzczMzMzMDEBxPQrXo3ANQBWuR+F6FA5AuB6F61G4DkBcj8L1KFwPQAAAAAAAABBA"},"y":{"dtype":"f8","bdata":"AAAAAAAAAAB7FK5H4Xq0P3sUrkfhesQ\u002fuB6F61G4zj97FK5H4XrUP5qZmZmZmdk\u002fuB6F61G43j\u002fsUbgehevhP3sUrkfheuQ\u002fCtejcD0K5z+amZmZmZnpPylcj8L1KOw\u002fuB6F61G47j+kcD0K16PwP+xRuB6F6\u002fE\u002fMzMzMzMz8z97FK5H4Xr0P8P1KFyPwvU\u002fCtejcD0K9z9SuB6F61H4P5qZmZmZmfk\u002f4XoUrkfh+j8pXI\u002fC9Sj8P3E9CtejcP0\u002fuB6F61G4\u002fj8AAAAAAAAAQKRwPQrXowBASOF6FK5HAUDsUbgehesBQI\u002fC9ShcjwJAMzMzMzMzA0DXo3A9CtcDQHsUrkfhegRAH4XrUbgeBUDD9Shcj8IFQGdmZmZmZgZACtejcD0KB0CuR+F6FK4HQFK4HoXrUQhA9ihcj8L1CECamZmZmZkJQD4K16NwPQpA4XoUrkfhCkCF61G4HoULQClcj8L1KAxAzczMzMzMDEBxPQrXo3ANQBWuR+F6FA5AuB6F61G4DkBcj8L1KFwPQAAAAAAAABBA"},"z":{"dtype":"f8","bdata":"cD0K16NwzT8nUgvVG8\u002fSPzyyNoM629Y\u002fzvnrf0jS2j97B0kXaqreP+ku3edGLeE\u002fCfjI21Pt4j8QnyYC85HkP4XLtNVuGOY\u002fgVSQ6NZ+5z+7DKuSDMToP6BlbETK5+k\u002fJoqYTaXq6j\u002f5C98MCs7rPx+2AqEylOw\u002faHF5WBhA7T+xbcI+YNXtP+wuoUZDWO4\u002fws8crXLN7j9VJdhJ+TnvP6\u002fab5Uao+8\u002fktbBHhgH8D+g48KTQ0DwP\u002fl4kGOef\u002fA\u002fatdx1Y7H8D9iofgMQxrxPzIN+8eiefE\u002fNJHt6ELn8T+8KxcfW2TyPyLiQ\u002fC98fI\u002fbjYHWNOP8z8787kflj70P1VPkgCU\u002ffQ\u002fLnL3jvDL9T9NHPnbaqj2P4fJGKhlkfc\u002fXwTH8fGE+D\u002fPoYua24D5P0sVJ9C3gvo\u002fQ1Bv2\u002fSH+z8KMIft6o38P6IbkH3tkf0\u002fOZ9ExVyR\u002fj\u002fmriDqton\u002fPwL4bjFUPABA8gMFlw2uAEAB4xw7IhkBQJO7USXYfAFA08W8j6XYAUAIb\u002fFrNCwCQEIxxJVkdwJAMiZhxHzIzj9+op1YBXfTP2ZoLVDhedc\u002fnK1bYqli2z8qmYky3yffP75+1IF2YOE\u002fADJR7i4T4z9G1CcmB6nkP5tDEUeqH+Y\u002fHM54hI915z+WHj8wBKroP4XVOkswvek\u002fdTudfxWw6j9U3OmLiITrP+WJjEklPew\u002fU3JcnT3d7D9rDZ7Bw2jtP2wYf3cw5O0\u002fMs3QyGVU7j9nyj0Zj77uP5rHz1f\u002fJ+8\u002fjCzaLQ6W7z826XyH+gbwPxwzUodWSvA\u002fq\u002fRaLWeX8D9R1dQXOfDwP0UNfgWMVvE\u002fhvZ+ZcfL8T+R4dgo8VDyPxEFZhGn5vI\u002fU+wsqhqN8z\u002fhKGIDEET0P4YTXzrfCvU\u002fBFCKw3jg9T9aKSZabMP2P9Kop2jysfc\u002f1b8QrPep+D+DpzHGKqn5P94cEWYLrfo\u002fcAtbo\u002fqy+z9jY90hTLj8PzmO\u002foxXuv0\u002faqLZ+Im2\u002fj8JzFi3dqr\u002fP1j8nJnzSQBAzqWmtHS4AEAkNqNQ7h8BQJImdwm+fwFACtMLmXPXAUCsIIq10yYCQOj2E5XZbQJAPESRsbCLzz8yAvQG99HTP8VJqQMFydc\u002fclcGmyah2z\u002fKrXdTSlHfPwWgpx2yaOE\u002fC0cFRFEN4z+Cc5MXzZPkP338bio1+uU\u002ftrSJ1Go\u002f5z+cDUuGKGPoPyIyd48DZuk\u002f9bO9TmhJ6j8ZXuHikA\u002frP2UZWJp2u+s\u002frRWhgL5Q7D\u002fn1n+IodPsP793++7QSO0\u002fUM22i1e17T+rgk7XeB7uPx9VYn+Oie4\u002fO29kaeX77j\u002fvmf8Im3rvP2crYfY9BfA\u002fYPXnLfJX8D8wYeroUbfwPzLl3AnyJPE\u002fuH8GQAqi8T8gNjMRbS\u002fyP2yK9ni

In [3]:
levels=np.arange(.25,2.01,.25)
sensitivity=pd.DataFrame({'water_m':levels,'cells_below_plane_percent':[100*np.mean(terrain<=level) for level in levels]})
display(sensitivity)
print('This is the share of sampled grid cells below a plane, not a validated inundated area.')

,water_m,cells_below_plane_percent
0,0.25,5.728566
1,0.50,19.377163
2,0.75,34.948097
3,1.00,46.789696
4,1.25,55.978470
5,1.50,66.589773
6,1.75,81.161092
7,2.00,94.502115


This is the share of sampled grid cells below a plane, not a validated inundated area.


In [4]:
dashboard(view='terrain',dataset='synthetic')

Open full-screen dashboard

## Lab: make the missing physics visible

Add an elevated ridge to the surface. Compare a simple below-water mask to a connected-ocean mask using a flood-fill starting from the left edge. The second mask should exclude isolated inland depressions until a connected path exists. Neither method supplies a full hydrodynamic model.

**Worked approach:** Use a queue; initialize ocean-edge cells below the water level; visit adjacent below-water cells until no new cells remain. Keep cell resolution fixed. Quantify the difference between all low cells and connected low cells.

**Deliverable:** Two labelled masks and a 3D scene with an explicit vertical-exaggeration note. List the data needed for a real analysis: surveyed heights, datum transformation, local coastal water levels, defences, connectivity and validation events.

Source context: [CoastalDEM v2.1](https://www.climatecentral.org/coastaldem-v2.1). No CoastalDEM raster is distributed or simulated by this surface.